# Exploración de disponibilidad de datos — Market Concentration Index (WITS)

**TFM:** Multicausalidad de la inflación: un estudio comparativo con técnicas de Machine Learning

**Objetivo de este notebook:**
Evaluar la calidad y disponibilidad real de `market_concentration_base.xlsx` (el HH Market
Concentration Index extraído en el notebook 08) antes de incorporarlo en el proyecto. 
Aacá hay **un único indicador**, por lo que el análisis se adapta: en vez de comparar disponibilidad
*entre* indicadores, el foco es diagnosticar la cobertura temporal y geográfica de *este* indicador,
y su naturaleza real de datos faltantes.

**Restricción de alcance:** el análisis se limita a los países que son **miembros de la ONU**
(según `df_regiones_miembros_onu.xlsx`, 193 unidades). No se consideran microestados no soberanos,
territorios ni dependencias (ej. Aruba, Bermudas, Islas Caimán) ni agregados regionales que WITS
incluye como "reporter" (ej. `Wld` = mundo, `BLX` = Bélgica-Luxemburgo agregado). Esta restricción
usa como filtro la pertenencia a la ONU, no un criterio de tamaño poblacional.




# Sección 1: Datos completos de Market Concentration

## Bloque 1 — Importación de librerías



In [23]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go

# Acumuladores para el reporte, igual que en el notebook 02
figuras_reporte = []
tablas_reporte = []


## Bloque 2 — Carga de datos

**Objetivo:** cargar el panel bruto de Market Concentration (notebook 08) y la tabla de
regiones/miembros ONU (notebook 03), sin modificar ninguno de los dos archivos originales.


**Resultado esperado:** dos DataFrames, `df_market_concentration` (5.129 filas, 3 columnas) y
`df_onu_base` (193 filas).




In [24]:
df_market_concentration = pd.read_excel("market_concentration_base.xlsx")
df_onu_base = pd.read_excel("df_regiones_miembros_onu.xlsx")

print("df_market_concentration:", df_market_concentration.shape)
print(df_market_concentration.columns.tolist())
print()
print("df_onu_base:", df_onu_base.shape)
print(df_onu_base.columns.tolist())


df_market_concentration: (5129, 3)
['reporter_code', 'year', 'hh_market_concentration']

df_onu_base: (193, 13)
['Member State', 'M49_country', 'ISO-alpha3', 'Other Names', 'Country or Area', 'M49_region', 'Region Name', 'M49_subregion', 'Sub-region Name', 'ISO-alpha2 Code', 'nombre_norm_principal', 'nombre_norm_alt', 'cow_code_countryVdem']


# Sección 2: Filtro y match con miembros ONU

## Bloque 3 — Match bidireccional de países (antes de cualquier análisis)

**Objetivo:** verificar, en los dos sentidos, la correspondencia entre los códigos `reporter_code`
de `market_concentration_base` y los códigos `ISO-alpha3` de los 193 miembros ONU — replicando la
misma lógica de verificación aplicada a V-Dem (`cow_code_countryVdem`) en el notebook 06.

**Justificación metodológica:** antes de fusionar ambas fuentes hay que saber exactamente qué se
gana y qué se pierde en el cruce

**Resultado esperado:** dos listas — `codigos_mc_fuera_onu` (31 códigos esperados) y
`miembros_onu_sin_dato` (15 países esperados) — impresas con su nombre.

**Qué comprobar:**
- Que el conteo de coincidencias (intersección) sea 178.
- Revisa `miembros_onu_sin_dato`: varios de estos nombres podrían corresponder en realidad a
  códigos legacy de WITS presentes en `codigos_mc_fuera_onu` bajo otro código (ej. Serbia/SER,
  Rep. Dem. del Congo/ZAR, Montenegro/MNT, Timor-Leste/TMP, Rumania/ROM, Sudán del Sur/SUD).
  Esto **no se corrige acá** — se documenta como hallazgo para una eventual estandarización.


In [25]:
codigos_mc = set(df_market_concentration["reporter_code"].unique())
codigos_onu = set(df_onu_base["ISO-alpha3"])

# Dirección 1: códigos de market_concentration que NO son miembros ONU
codigos_mc_fuera_onu = sorted(codigos_mc - codigos_onu)

# Dirección 2: miembros ONU sin ninguna fila en market_concentration
miembros_onu_sin_dato = sorted(codigos_onu - codigos_mc)

print(f"Códigos únicos en market_concentration: {len(codigos_mc)}")
print(f"Miembros ONU (universo de referencia): {len(codigos_onu)}")
print(f"Coincidencias en ambos sentidos: {len(codigos_mc & codigos_onu)} de 193")
print()

print(f"[Dirección 1] Códigos de market_concentration que NO son miembros ONU ({len(codigos_mc_fuera_onu)}):")
print(codigos_mc_fuera_onu)
print()

onu_idx = df_onu_base.set_index("ISO-alpha3")
print(f"[Dirección 2] Miembros ONU sin ninguna fila en market_concentration ({len(miembros_onu_sin_dato)}):")
print(onu_idx.loc[miembros_onu_sin_dato, "Country or Area"].to_string())


Códigos únicos en market_concentration: 209
Miembros ONU (universo de referencia): 193
Coincidencias en ambos sentidos: 178 de 193

[Dirección 1] Códigos de market_concentration que NO son miembros ONU (31):
['ABW', 'AIA', 'ANT', 'BLX', 'BMU', 'COK', 'CUW', 'CYM', 'FRO', 'GLP', 'GRL', 'GUF', 'HKG', 'MAC', 'MNT', 'MSR', 'MTQ', 'MYT', 'NCL', 'OAS', 'PSE', 'PYF', 'REU', 'ROM', 'SER', 'SUD', 'TCA', 'TMP', 'WLF', 'Wld', 'ZAR']

[Dirección 2] Miembros ONU sin ninguna fila en market_concentration (15):
ISO-alpha3
COD         Democratic Republic of the Congo
GNQ                        Equatorial Guinea
HTI                                    Haiti
LIE                            Liechtenstein
MCO                                   Monaco
MHL                         Marshall Islands
MNE                               Montenegro
NRU                                    Nauru
PRK    Democratic People's Republic of Korea
ROU                                  Romania
SMR                               San 

Chequeamos manualmente esos países faltantes y encontramos que algunos sí estan incluidos pero con pequeñas diferencias en la codificación. 
Esto es porque Market Concentration no utiliza exclusivamente ISO alpha-3. Utiliza una mezcla de códigos ISO actuales y códigos históricos ("legacy codes") heredados de UN COMTRADE/WITS, muchas veces para preservar la continuidad histórica de las series . La pagina oficial de Trade comision proporciona una tabla con estas equivalencias, así que aquí la utilizamos para incluir en el anaálisis a países que efectivamente esta, pero con otra codificacion

In [26]:
# Objetivo: recodificar códigos históricos/legacy de WITS a su ISO3 vigente, usando
# la tabla de correspondencia oficial de USITC (Dynamic Gravity / Comtrade / WITS).
# Fuente: https://www.usitc.gov/faq/question/how_do_i_merge_your_gravity_data_comtrade_or_wits.htm

df_codigos_historicos = pd.read_excel("estandarizar_iso3_codhistorico.xlsx", sheet_name="codigos_historicos")

# Solo se usan las filas con equivalente ISO3 real en la columna 'Comtrade' (se excluyen
# entidades sin ISO3 vigente: Neutral Zone, Sikkim, Taiwan, US Misc. Pacific Islands)
mapa_valido = df_codigos_historicos[
    df_codigos_historicos["Comtrade"].str.match(r"^[A-Z]{3}$", na=False)
]
codigos_legacy_a_iso3 = dict(zip(mapa_valido["WITS"], mapa_valido["Comtrade"]))

codigos_legacy_presentes = set(df_market_concentration["reporter_code"]) & set(codigos_legacy_a_iso3)
n_filas_afectadas = df_market_concentration["reporter_code"].isin(codigos_legacy_a_iso3).sum()

df_market_concentration["reporter_code"] = df_market_concentration["reporter_code"].replace(codigos_legacy_a_iso3)

print(f"Códigos legacy presentes en la base y recodificados: {sorted(codigos_legacy_presentes)}")
print(f"Filas recodificadas: {n_filas_afectadas}")
print(f"Códigos únicos después de la recodificación: {df_market_concentration['reporter_code'].nunique()}")

# Verificación: no deben quedar códigos legacy, y cada código legacy presente debe
# haberse fusionado en su ISO3 vigente (ya se confirmó que no hay solapamiento de años
# entre código legacy y código vigente para ningún par, por lo tanto no hay duplicados)
dups = df_market_concentration.duplicated(subset=["reporter_code", "year"]).sum()
print(f"Duplicados (reporter_code, year) tras la fusión: {dups}")
assert dups == 0, "Hay duplicados país-año tras la fusión — revisar antes de continuar"
assert not df_market_concentration["reporter_code"].isin(codigos_legacy_a_iso3.keys()).any()
print("Verificación OK.")

Códigos legacy presentes en la base y recodificados: ['MNT', 'ROM', 'SER', 'SUD', 'TMP', 'ZAR']
Filas recodificadas: 106
Códigos únicos después de la recodificación: 208
Duplicados (reporter_code, year) tras la fusión: 0
Verificación OK.


In [27]:
# Objetivo: verificar el match bidireccional después de la recodificación de códigos legacy.
codigos_mc_actualizado = set(df_market_concentration["reporter_code"].unique())
codigos_mc_fuera_onu_actualizado = sorted(codigos_mc_actualizado - codigos_onu)
miembros_onu_sin_dato_actualizado = sorted(codigos_onu - codigos_mc_actualizado)

print(f"Coincidencias tras la recodificación: {len(codigos_mc_actualizado & codigos_onu)} de 193")
print(f"Códigos MC fuera de ONU (tras recodificación): {len(codigos_mc_fuera_onu_actualizado)}")
print(f"Miembros ONU sin dato (tras recodificación): {len(miembros_onu_sin_dato_actualizado)}")
print(onu_idx.loc[miembros_onu_sin_dato_actualizado, "Country or Area"].to_string())

Coincidencias tras la recodificación: 183 de 193
Códigos MC fuera de ONU (tras recodificación): 25
Miembros ONU sin dato (tras recodificación): 10
ISO-alpha3
GNQ                        Equatorial Guinea
HTI                                    Haiti
LIE                            Liechtenstein
MCO                                   Monaco
MHL                         Marshall Islands
NRU                                    Nauru
PRK    Democratic People's Republic of Korea
SMR                               San Marino
SOM                                  Somalia
SSD                              South Sudan


## Bloque 4 — Construcción del DataFrame con columnas de región

**Objetivo:** unir `df_market_concentration` con `df_onu_base` (inner join, `ISO-alpha3` ==
`reporter_code`), agregando las columnas de región/subregión, y **restringiendo el análisis a los
países que matchean** (miembros ONU con al menos un dato).



**Resultado esperado:** `df_market_concentration_onu`


In [28]:
df_market_concentration_onu = df_onu_base.merge(
    df_market_concentration,
    left_on="ISO-alpha3",
    right_on="reporter_code",
    how="inner",
)

cols_region = [
    "Country or Area", "ISO-alpha3", "ISO-alpha2 Code",
    "M49_region", "Region Name", "M49_subregion", "Sub-region Name",
]
df_market_concentration_onu = df_market_concentration_onu[
    cols_region + ["year", "hh_market_concentration"]
]

print("Shape df_market_concentration_onu:", df_market_concentration_onu.shape)
print(f"Países únicos: {df_market_concentration_onu['ISO-alpha3'].nunique()} de 193")
df_market_concentration_onu.head()


Shape df_market_concentration_onu: (4760, 9)


Países únicos: 183 de 193


,Country or Area,ISO-alpha3,ISO-alpha2 Code,M49_region,Region Name,M49_subregion,Sub-region Name,year,hh_market_concentration
0,United States of America,USA,US,19,Americas,21,Northern America,1991,0.120666
1,United States of America,USA,US,19,Americas,21,Northern America,1992,0.097600
2,United States of America,USA,US,19,Americas,21,Northern America,1993,0.087233
3,United States of America,USA,US,19,Americas,21,Northern America,1994,0.076783
4,United States of America,USA,US,19,Americas,21,Northern America,1995,0.068333


## Bloque 5 — Descripción general del panel

**Objetivo:** revisar la forma general del panel restringido a ONU: rango de años, y estadísticos
descriptivos del propio índice.

**Justificación metodológica:** antes de evaluar disponibilidad conviene conocer el rango de valores
esperado del indicador. El HH Market Concentration Index está acotado teóricamente en [0, 1]
(mayor valor = mayor concentración de exportaciones); valores fuera de ese rango indicarían un
problema de extracción, no de disponibilidad.

**Resultado esperado:** impresión de shape, rango de años, y `describe()` del indicador.

**Qué comprobar:** que min/max de `hh_market_concentration` estén dentro de [0, 1]. Si no,
revisar el notebook 08 (posible error de parseo del JSON).


In [29]:
print("Shape:", df_market_concentration_onu.shape)
print("Rango de años:", df_market_concentration_onu["year"].min(), "-", df_market_concentration_onu["year"].max())
print()
print(df_market_concentration_onu["hh_market_concentration"].describe())


Shape: (4760, 9)
Rango de años: 1988 - 2023

count    4760.000000
mean        0.160227
std         0.136708
min         0.029960
25%         0.073745
50%         0.108286
75%         0.192796
max         0.991874
Name: hh_market_concentration, dtype: float64


# Sección 3: Análisis de recorte temporal

## Bloque 6 — Cobertura temporal por país

**Objetivo:** para cada uno de los 178 países, calcular cuántos años tiene reportados, y el primer
y último año disponible.

**Justificación metodológica:** con un solo indicador, la pregunta relevante de "disponibilidad"
ya no es "¿qué indicadores sobreviven?" (como en el 02) sino "¿qué tan completa es la serie de
*cada país* para este indicador?". Esto es clave porque el panel del TFM es país-año: un país con
solo 1-2 años reportados aporta muy poca información a un clustering longitudinal.

**Resultado esperado:** `cobertura_pais`, con columnas `n_anios`, `anio_min`, `anio_max`, ordenado
de menor a mayor cobertura.


In [30]:
cobertura_pais = (
    df_market_concentration_onu
    .groupby(["ISO-alpha3", "Country or Area"])["year"]
    .agg(n_anios="count", anio_min="min", anio_max="max")
    .reset_index()
    .sort_values("n_anios")
)

print(cobertura_pais["n_anios"].describe())
print()
print("Países con menor cobertura temporal:")
cobertura_pais.head(10)


count    183.000000
mean      26.010929
std        8.230197
min        1.000000
25%       24.000000
50%       29.000000
75%       31.000000
max       36.000000
Name: n_anios, dtype: float64

Países con menor cobertura temporal:


,ISO-alpha3,Country or Area,n_anios,anio_min,anio_max
157,TCD,Chad,1,1995,1995
78,IRQ,Iraq,1,2014,2014
52,ERI,Eritrea,1,2003,2003
161,TKM,Turkmenistan,4,1997,2000
45,DJI,Djibouti,4,2009,2023
173,UZB,Uzbekistan,7,2017,2023
146,SLE,Sierra Leone,7,2000,2018
95,LBR,Liberia,7,2017,2023
96,LBY,Libya,8,2007,2019
177,VUT,Vanuatu,8,1993,2011


## Bloque 7 — Cobertura de países por año

**Objetivo:** para cada año del rango observado, contar cuántos países tienen datos

**Justificación metodológica:** complementa el Bloque 6 (cobertura por país) con la vista inversa
(cobertura por año). Es relevante para decidir el recorte temporal final del panel: años con muy
pocos países reportando limitan la comparabilidad transversal en ese corte de tiempo.

**Resultado esperado:** serie `cobertura_anual`, y gráfico de línea de la cantidad de países con
dato por año.



In [31]:
cobertura_anual = (
    df_market_concentration_onu.groupby("year")["ISO-alpha3"].nunique().reset_index(name="n_paises")
)

fig_cobertura_anual = px.line(
    cobertura_anual, x="year", y="n_paises", markers=True,
    labels={"year": "Año", "n_paises": "Nº de países con dato"},
)
fig_cobertura_anual.update_layout(
    title="Cobertura de países ONU por año — Market Concentration Index",
    yaxis_range=[0, 193],
    margin=dict(b=90),
)
fig_cobertura_anual.add_hline(
    y=178, line_dash="dot", line_color="gray",
    annotation_text="Máximo posible (178 países con al menos un dato)",
)
fig_cobertura_anual.add_annotation(
    text="Nota: cantidad de países (universo ONU) con un dato del índice en cada año.",
    xref="paper", yref="paper", x=0, y=-0.2, showarrow=False,
    font=dict(size=12, color="gray"), align="left",
)

figuras_reporte.append(("cobertura_anual_market_concentration", fig_cobertura_anual))
fig_cobertura_anual.show()


In [32]:
n_paises_total_mc = df_market_concentration_onu["ISO-alpha3"].nunique()

paises_con_dato_anio_mc = (
    df_market_concentration_onu.dropna(subset=["hh_market_concentration"])
    .groupby("year")["ISO-alpha3"]
    .nunique()
    .rename("n_paises_con_dato")
)

anios_totales_mc = sorted(df_market_concentration_onu["year"].unique())
disponibilidad_anio_mc = pd.DataFrame(index=anios_totales_mc).join(paises_con_dato_anio_mc)
disponibilidad_anio_mc["n_paises_con_dato"] = disponibilidad_anio_mc["n_paises_con_dato"].fillna(0)

# Densidad real: % de celdas país-indicador efectivamente completas en cada año
# (aquí "indicador" = 1, así que equivale al % de países con dato, pero se mantiene
# la misma lógica de cálculo que en WDI y V-Dem para comparabilidad conceptual)
celdas_posibles_anio_mc = n_paises_total_mc * 1
celdas_con_dato_anio_mc = df_market_concentration_onu.dropna(subset=["hh_market_concentration"]).groupby("year").size()
disponibilidad_anio_mc["densidad_real"] = (
    celdas_con_dato_anio_mc.reindex(anios_totales_mc, fill_value=0).values / celdas_posibles_anio_mc
)

fig_densidad_anio_mc = px.line(
    disponibilidad_anio_mc.reset_index().rename(columns={"index": "year"}),
    x="year",
    y="densidad_real",
    markers=True,
    labels={"densidad_real": "% de celdas país-indicador con dato"},
)
fig_densidad_anio_mc.update_layout(
    title=dict(text="% de cobertura de datos a lo largo del tiempo (MC)", x=0.5, xanchor="center"),
    xaxis_title="Año",
    yaxis_title="% de celdas país-indicador con dato",
    yaxis_range=[0, 1],
    margin=dict(b=90),
)
fig_densidad_anio_mc.add_annotation(
    text="Nota: % de cobertura de datos a lo largo del tiempo (solo tenemos un indicador, así que se evalúa su % de presencia en los países).",
    xref="paper", yref="paper", x=0, y=-0.22, showarrow=False,
    font=dict(size=12, color="gray"), align="left",
)

figuras_reporte.append(("densidad_anio_mc", fig_densidad_anio_mc))
fig_densidad_anio_mc.show()

### Gráfico — Disponibilidad regional en el tiempo (heatmap)

**Objetivo:** ver si la cobertura del índice mejora de forma pareja entre regiones a lo largo del tiempo, o si hay regiones sistemáticamente rezagadas — mismo chequeo hecho para el WDI y para V-Dem.

In [33]:
n_paises_por_region_mc = df_onu_base.groupby("Region Name")["ISO-alpha3"].nunique()

anios_totales_mc = list(range(
    int(df_market_concentration_onu["year"].min()), int(df_market_concentration_onu["year"].max()) + 1
))

celdas_con_dato_region_anio_mc = (
    df_market_concentration_onu.groupby(["year", "Region Name"]).size().unstack("Region Name")
    .reindex(anios_totales_mc, fill_value=0)
)
densidad_region_anio_mc = celdas_con_dato_region_anio_mc.div(n_paises_por_region_mc, axis=1).fillna(0)
matriz_region_anio_mc = densidad_region_anio_mc.T

fig_heatmap_region_mc = px.imshow(
    matriz_region_anio_mc,
    labels=dict(x="Año", y="Región", color="% países con dato"),
    x=matriz_region_anio_mc.columns,
    y=matriz_region_anio_mc.index,
    color_continuous_scale="Blues",
    aspect="auto",
)
fig_heatmap_region_mc.update_layout(
    title="Disponibilidad regional del Market Concentration Index a lo largo del tiempo",
    margin=dict(b=90),
)
fig_heatmap_region_mc.add_annotation(
    text="Nota: % de los miembros ONU de cada región con un dato del índice, para cada año (sobre el total de miembros ONU de esa región, no solo los 178 con match).",
    xref="paper", yref="paper", x=0, y=-0.2, showarrow=False,
    font=dict(size=12, color="gray"), align="left",
)

figuras_reporte.append(("heatmap_region_anio_mc", fig_heatmap_region_mc))
fig_heatmap_region_mc.show()


In [34]:
fig_lineas_region_anio_mc = px.line(
    densidad_region_anio_mc.reset_index(),
    x="year",
    y=densidad_region_anio_mc.columns.tolist(),
    labels={"year": "Año", "value": "% de países con dato", "variable": "Región"},
)
fig_lineas_region_anio_mc.update_layout(
    title="Disponibilidad regional del Market Concentration Index a lo largo del tiempo (líneas)",
    yaxis_tickformat=".0%",
    legend_title_text="Región",
    margin=dict(b=90),
)
fig_lineas_region_anio_mc.add_annotation(
    text="Nota: mismos datos que el heatmap de arriba — % de los miembros ONU de cada región con un dato del índice, para cada año.",
    xref="paper", yref="paper", x=0, y=-0.2, showarrow=False,
    font=dict(size=12, color="gray"), align="left",
)

figuras_reporte.append(("lineas_region_anio_mc", fig_lineas_region_anio_mc))
fig_lineas_region_anio_mc.show()

### Gráfico — Curva de sensibilidad de la densidad al año de inicio del panel

**Objetivo:** misma lógica que en el WDI y V-Dem — para cada año de inicio candidato, calcular qué % del panel país-año (178 países × años) tiene dato, manteniendo fijo el año final en el último año observado.

In [35]:
anio_min_disponible_mc = int(df_market_concentration_onu["year"].min())
anio_max_disponible_mc = int(df_market_concentration_onu["year"].max())
n_paises_mc = df_market_concentration_onu["ISO-alpha3"].nunique()

anios_inicio_candidatos_mc = list(range(anio_min_disponible_mc, anio_max_disponible_mc + 1, 2))

def calcular_densidad_panel_mc(datos_mc, anio_inicio, anio_fin):
    sub = datos_mc[(datos_mc["year"] >= anio_inicio) & (datos_mc["year"] <= anio_fin)]
    n_celdas_totales = n_paises_mc * (anio_fin - anio_inicio + 1)
    n_celdas_con_dato = sub["hh_market_concentration"].notna().sum()
    return n_celdas_con_dato / n_celdas_totales

densidad_por_inicio_mc = [
    calcular_densidad_panel_mc(df_market_concentration_onu, anio_inicio, anio_max_disponible_mc)
    for anio_inicio in anios_inicio_candidatos_mc
]

fig_curva_sensibilidad_mc = px.line(
    x=anios_inicio_candidatos_mc, y=densidad_por_inicio_mc, markers=True,
    labels={"x": "Año de inicio candidato", "y": "Densidad del panel"},
)
fig_curva_sensibilidad_mc.add_vline(x=1990, line_dash="dash", line_color="gray")
fig_curva_sensibilidad_mc.update_layout(
    title="Sensibilidad de la densidad del panel al año de inicio (Market Concentration)",
    yaxis_tickformat=".0%",
    margin=dict(b=90),
)
fig_curva_sensibilidad_mc.add_annotation(
    text="Nota: cada punto es la densidad del panel (178 países × años) si arrancara en ese año, hasta el último año observado.",
    xref="paper", yref="paper", x=0, y=-0.2, showarrow=False,
    font=dict(size=12, color="gray"), align="left",
)

figuras_reporte.append(("curva_sensibilidad_anio_inicio_mc", fig_curva_sensibilidad_mc))
fig_curva_sensibilidad_mc.show()


## Bloque 8 — Cobertura por región (M49)

**Objetivo:** para cada región M49 (`Region Name`), calcular qué porcentaje de sus miembros ONU
tiene al menos un dato del indicador.

**Justificación metodológica:** una cobertura desigual por región es un riesgo metodológico
relevante para el TFM (comparación entre países). si, por ejemplo, Oceanía o África tienen mucha
menos cobertura que Europa, cualquier patrón que emerja en el clustering posterior podría reflejar
sesgo de disponibilidad de datos y no una diferencia estructural real. 




In [36]:
cobertura_region = (
    df_onu_base.groupby("Region Name")["ISO-alpha3"].nunique().rename("total_miembros_onu").to_frame()
    .join(
        df_market_concentration_onu.groupby("Region Name")["ISO-alpha3"].nunique().rename("con_al_menos_un_dato")
    )
)
cobertura_region["pct_cobertura"] = (
    cobertura_region["con_al_menos_un_dato"] / cobertura_region["total_miembros_onu"] * 100
).round(1)

cobertura_region = cobertura_region.sort_values("pct_cobertura")
cobertura_region


,total_miembros_onu,con_al_menos_un_dato,pct_cobertura
Region Name,,,
Oceania,14,12,85.7
Europe,43,40,93.0
Africa,54,51,94.4
Americas,35,34,97.1
Asia,47,46,97.9


In [37]:
fig_cobertura_region = px.bar(
    cobertura_region.reset_index(),
    x="Region Name", y="pct_cobertura",
    text="pct_cobertura",
    labels={"Region Name": "Región (M49)", "pct_cobertura": "% de miembros ONU con al menos un dato"},
)
fig_cobertura_region.update_traces(texttemplate="%{text}%", textposition="outside")
fig_cobertura_region.update_layout(
    title="Cobertura del Market Concentration Index por región ONU (M49)",
    yaxis_range=[0, 105],
    margin=dict(b=90),
)
fig_cobertura_region.add_annotation(
    text="Nota: % de los miembros ONU de cada región con al menos un dato del índice, en cualquier año del panel.",
    xref="paper", yref="paper", x=0, y=-0.2, showarrow=False,
    font=dict(size=12, color="gray"), align="left",
)

figuras_reporte.append(("cobertura_region_market_concentration", fig_cobertura_region))
fig_cobertura_region.show()


# Sección 4: Análisis de dispersión 

A continuación vemos como se modifica la dispersión de datos si recortamos entre 1990 y 2024

In [38]:
def calcular_cobertura_por_pais_mc(datos_mc, anio_inicio, anio_fin, universo_paises):
    sub = datos_mc[(datos_mc["year"] >= anio_inicio) & (datos_mc["year"] <= anio_fin)]
    n_anios_totales = anio_fin - anio_inicio + 1
    anios_con_dato = sub.groupby("ISO-alpha3")["year"].nunique()
    cobertura = (
        pd.Series(index=universo_paises, dtype=float)
        .fillna(0)
        .add(anios_con_dato.reindex(universo_paises).fillna(0), fill_value=0)
        / n_anios_totales
    )
    return cobertura

universo_paises_mc = df_market_concentration_onu["ISO-alpha3"].unique()

In [39]:
cobertura_pct_completo_mc = calcular_cobertura_por_pais_mc(
    df_market_concentration_onu, anio_min_disponible_mc, anio_max_disponible_mc, universo_paises_mc
)
cobertura_pct_2000_2020_mc = calcular_cobertura_por_pais_mc(
    df_market_concentration_onu, 2000, 2020, universo_paises_mc
)

etiqueta_rango_completo = f"Rango completo ({anio_min_disponible_mc}-{anio_max_disponible_mc})"

comparacion_dispersion_pais_mc = pd.DataFrame({
    etiqueta_rango_completo: cobertura_pct_completo_mc.describe(),
    "2000-2020": cobertura_pct_2000_2020_mc.describe(),
})
comparacion_dispersion_pais_mc.loc["IQR"] = comparacion_dispersion_pais_mc.loc["75%"] - comparacion_dispersion_pais_mc.loc["25%"]
comparacion_dispersion_pais_mc.loc["CV"] = comparacion_dispersion_pais_mc.loc["std"] / comparacion_dispersion_pais_mc.loc["mean"]

tablas_reporte.append(("dispersion_pais_densidad_mc", comparacion_dispersion_pais_mc))

comparacion_dispersion_pais_mc

,Rango completo (1988-2023),2000-2020
count,183.000000,183.000000
mean,0.722526,0.877700
std,0.228617,0.239995
min,0.027778,0.000000
25%,0.666667,0.880952
50%,0.805556,1.000000
75%,0.861111,1.000000
max,1.000000,1.000000
IQR,0.194444,0.119048
CV,0.316413,0.273436


In [40]:
df_dispersion_pais_mc = pd.concat([
    pd.DataFrame({"cobertura": cobertura_pct_completo_mc, "Escenario": etiqueta_rango_completo}),
    pd.DataFrame({"cobertura": cobertura_pct_2000_2020_mc, "Escenario": "2000-2020"}),
])

fig_dispersion_pais_mc = px.box(
    df_dispersion_pais_mc, x="Escenario", y="cobertura", points="outliers",
    labels={"cobertura": "% de años con dato"},
)
fig_dispersion_pais_mc.update_layout(
    title=f"Dispersión de cobertura por país: {etiqueta_rango_completo} vs. 2000-2020 (Market Concentration)",
    margin=dict(b=90),
)
fig_dispersion_pais_mc.add_annotation(
    text="Nota: para cada país, % de los años del rango correspondiente con un dato del índice.",
    xref="paper", yref="paper", x=0, y=-0.2, showarrow=False,
    font=dict(size=12, color="gray"), align="left",
)

figuras_reporte.append(("dispersion_pais_densidad_mc", fig_dispersion_pais_mc))
fig_dispersion_pais_mc.show()

## 4.2 Oceanía: cobertura por país

A diferencia de V-Dem (donde 8 de los 14 miembros ONU de Oceanía ni siquiera están en la base), acá **12 de los 14 están presentes** — faltan Islas Marshall y Nauru. Veamos cómo se distribuye la cobertura temporal entre los 12 que sí tienen datos.

In [41]:
cobertura_oceania_mc = (
    df_market_concentration_onu[df_market_concentration_onu["Region Name"] == "Oceania"]
    .groupby("Country or Area")["year"].nunique()
    .rename("n_anios")
    .reset_index()
)
cobertura_oceania_mc["pct_anios"] = cobertura_oceania_mc["n_anios"] / (anio_max_disponible_mc - anio_min_disponible_mc + 1)
cobertura_oceania_mc = cobertura_oceania_mc.sort_values("pct_anios", ascending=False)

fig_cobertura_oceania_mc = px.bar(
    cobertura_oceania_mc,
    x="Country or Area", y="pct_anios",
    labels={"pct_anios": "% de años con dato", "Country or Area": "País"},
)
fig_cobertura_oceania_mc.update_layout(
    title="Cobertura temporal por país — Oceanía en Market Concentration (12 de 14 miembros ONU presentes)",
    yaxis_tickformat=".0%",
    margin=dict(b=110),
)
fig_cobertura_oceania_mc.add_annotation(
    text="Nota: Islas Marshall y Nauru no tienen ninguna fila en la base y no aparecen en este gráfico.",
    xref="paper", yref="paper", x=0, y=-0.4, showarrow=False,
    font=dict(size=12, color="gray"), align="left",
)

figuras_reporte.append(("cobertura_oceania_mc", fig_cobertura_oceania_mc))
fig_cobertura_oceania_mc.show()


# Bloque de exportación

Todo lo de acá abajo está comentado por defecto — descomentar solo los bloques que se necesiten antes de correr.

In [42]:
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path_mc = OUT_DIR / "disponibilidad_market_concentration.xlsx"

match_no_onu_df = pd.DataFrame({"reporter_code_no_onu": codigos_mc_fuera_onu_actualizado})
match_onu_sin_dato_df = onu_idx.loc[miembros_onu_sin_dato_actualizado, ["Country or Area"]].reset_index()

with pd.ExcelWriter(out_path_mc) as writer:
    cobertura_pais.to_excel(writer, sheet_name="cobertura_pais", index=False)
    cobertura_anual.to_excel(writer, sheet_name="cobertura_anual", index=False)
    cobertura_region.to_excel(writer, sheet_name="cobertura_region")
    match_no_onu_df.to_excel(writer, sheet_name="match_no_onu", index=False)
    match_onu_sin_dato_df.to_excel(writer, sheet_name="match_onu_sin_dato", index=False)

print(f"Exportado: {out_path_mc}")


Exportado: data\processed\disponibilidad_market_concentration.xlsx


In [43]:
with pd.ExcelWriter(out_path_mc, mode="a", engine="openpyxl", if_sheet_exists="replace") as writer:
    comparacion_dispersion_pais_mc.to_excel(writer, sheet_name="dispersion_pais_densidad")

print(f"Hoja de dispersión añadida a: {out_path_mc}")


Hoja de dispersión añadida a: data\processed\disponibilidad_market_concentration.xlsx


In [44]:
import re
import nbformat

NOTEBOOK_PATH = "09_disponibilidad_datos_MC.ipynb"  # ajustar si el nombre/ruta difiere

figuras_por_nombre = dict(figuras_reporte)
tablas_por_nombre = dict(tablas_reporte)

def markdown_a_html(texto):
    """Conversor liviano de Markdown a HTML (headers, negrita, código inline, listas, párrafos)."""
    lineas = texto.split("\n")
    html = []
    en_lista = False
    for linea in lineas:
        l = linea.rstrip()
        if l.startswith("### "):
            if en_lista:
                html.append("</ul>"); en_lista = False
            html.append(f"<h3>{l[4:]}</h3>")
        elif l.startswith("## "):
            if en_lista:
                html.append("</ul>"); en_lista = False
            html.append(f"<h2>{l[3:]}</h2>")
        elif l.startswith("# "):
            if en_lista:
                html.append("</ul>"); en_lista = False
            html.append(f"<h1>{l[2:]}</h1>")
        elif l.startswith("- "):
            if not en_lista:
                html.append("<ul>"); en_lista = True
            html.append(f"<li>{l[2:]}</li>")
        elif l.strip() == "":
            if en_lista:
                html.append("</ul>"); en_lista = False
            html.append("")
        else:
            if en_lista:
                html.append("</ul>"); en_lista = False
            html.append(f"<p>{l}</p>")
    if en_lista:
        html.append("</ul>")
    texto_html = "\n".join(html)
    texto_html = re.sub(r"\*\*(.+?)\*\*", r"<strong>\1</strong>", texto_html)
    texto_html = re.sub(r"`(.+?)`", r"<code>\1</code>", texto_html)
    return texto_html

nb_en_disco = nbformat.read(NOTEBOOK_PATH, as_version=4)

partes_html = []
nombres_incrustados = set()

for cell in nb_en_disco.cells:
    if cell.cell_type == "markdown":
        partes_html.append(markdown_a_html(cell.source))
    elif cell.cell_type == "code":
        for nombre in re.findall(r'figuras_reporte\.append\(\(\s*"([^"]+)"', cell.source):
            if nombre in figuras_por_nombre:
                partes_html.append(f"<h4>Figura: {nombre}</h4>")
                partes_html.append(figuras_por_nombre[nombre].to_html(full_html=False, include_plotlyjs="cdn"))
                nombres_incrustados.add(("figura", nombre))
        for nombre in re.findall(r'tablas_reporte\.append\(\(\s*"([^"]+)"', cell.source):
            if nombre in tablas_por_nombre:
                partes_html.append(f"<h4>Tabla: {nombre}</h4>")
                partes_html.append(tablas_por_nombre[nombre].to_html())
                nombres_incrustados.add(("tabla", nombre))

html_path = OUT_DIR / "reporte_disponibilidad_mc.html"
with open(html_path, "w", encoding="utf-8") as f:
    f.write("<html><head><meta charset='utf-8'><title>Reporte de disponibilidad Market Concentration</title></head><body>\n")
    f.write("\n".join(partes_html))
    f.write("\n</body></html>")

figuras_faltantes = set(figuras_por_nombre) - {n for t, n in nombres_incrustados if t == "figura"}
tablas_faltantes = set(tablas_por_nombre) - {n for t, n in nombres_incrustados if t == "tabla"}

print(f"Exportado: {html_path}")
print(f"Figuras incrustadas: {len(figuras_por_nombre) - len(figuras_faltantes)} / {len(figuras_por_nombre)}")
print(f"Tablas incrustadas: {len(tablas_por_nombre) - len(tablas_faltantes)} / {len(tablas_por_nombre)}")
if figuras_faltantes:
    print(f"⚠️ Figuras en memoria pero no encontradas en el .ipynb guardado: {figuras_faltantes}")
if tablas_faltantes:
    print(f"⚠️ Tablas en memoria pero no encontradas en el .ipynb guardado: {tablas_faltantes}")


Exportado: data\processed\reporte_disponibilidad_mc.html
Figuras incrustadas: 8 / 8
Tablas incrustadas: 1 / 1
